### Model Training v2

Loads the existing FastText embeddings and preprocessed data, then trains the model with improvements to boost fake-class recall.

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from gensim.models import FastText
from nltk.tokenize import word_tokenize
from torch.utils.data import DataLoader
from dataset_class.job_post_dataset import JobPostingDataset
from model_construction.model import FakeJobDetector
import os, json, pickle
import math


#Load preprocessed data
combined_df = pd.read_csv("./data/clean/fake_job_postings_ALL.csv")

numeric_cols = [
    "telecommuting", "missing_count", "total_text_len", "company_profile_len",
    "description_len", "requirements_len", "benefits_len",
    "company_profile_word_count", "description_word_count",
    "requirements_word_count", "benefits_word_count",
    "salary_provided", "has_company_profile", "vague_location",
    "has_company_logo", "has_questions"
]

print(f"Dataset: {combined_df.shape[0]} rows, {combined_df.shape[1]} cols")
print(f"Class balance: {combined_df['fraudulent'].value_counts().to_dict()}")

In [ ]:
#Train / Val / Test split (80/10/10, stratified so same proportion of fraudulent vs non-fraudulent in each set)
train_data, temp_data = train_test_split(
    combined_df, test_size=0.2, random_state=42, stratify=combined_df['fraudulent']
)
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, random_state=42, stratify=temp_data['fraudulent']
)

X_train_text    = train_data['full_text'].tolist()
X_train_numeric = train_data[numeric_cols].values.astype(np.float32)
y_train         = train_data['fraudulent'].values.tolist()

X_val_text      = val_data['full_text'].tolist()
X_val_numeric   = val_data[numeric_cols].values.astype(np.float32)
y_val           = val_data['fraudulent'].values.tolist()

X_test_text     = test_data['full_text'].tolist()
X_test_numeric  = test_data[numeric_cols].values.astype(np.float32)
y_test          = test_data['fraudulent'].values.tolist()

print(f"Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}")

In [ ]:
#Scale non-binary numeric features
non_binary_cols = [
    "missing_count", "total_text_len", "company_profile_len", "description_len",
    "requirements_len", "benefits_len", "company_profile_word_count",
    "description_word_count", "requirements_word_count", "benefits_word_count"
]
non_binary_indices = [numeric_cols.index(col) for col in non_binary_cols]

scaler = StandardScaler()
X_train_numeric[:, non_binary_indices] = scaler.fit_transform(X_train_numeric[:, non_binary_indices])
X_val_numeric[:,   non_binary_indices] = scaler.transform(X_val_numeric[:,   non_binary_indices])
X_test_numeric[:,  non_binary_indices] = scaler.transform(X_test_numeric[:,  non_binary_indices])

In [ ]:
#Load pretrained FastText embeddings
fasttext_model = FastText.load("./optimal_fasttext.bin")
print(f"FastText vocab: {len(fasttext_model.wv)} words, {fasttext_model.wv.vector_size}-dim")

In [ ]:
CACHE_DIR = "./data/cache"
os.makedirs(CACHE_DIR, exist_ok=True)

tok_cache   = os.path.join(CACHE_DIR, "tokenized_splits.pkl")
vocab_cache = os.path.join(CACHE_DIR, "vocab.json")

if os.path.exists(tok_cache) and os.path.exists(vocab_cache):
    #Load cached tokenized data
    print("Loading cached tokenized data...")
    with open(tok_cache, "rb") as f:
        cached = pickle.load(f)
    train_texts_tok = cached["train"]
    val_texts_tok   = cached["val"]
    test_texts_tok  = cached["test"]

    with open(vocab_cache, "r") as f:
        vocab = json.load(f)

    print(f"Vocab size: {len(vocab)} (loaded from cache)")
else:
    #Tokenize and encode from scratch
    print("Tokenizing (first run, will be cached)...")
    train_tk = [word_tokenize(text.lower()) for text in X_train_text]
    val_tk   = [word_tokenize(text.lower()) for text in X_val_text]
    test_tk  = [word_tokenize(text.lower()) for text in X_test_text]

    # Build vocab from FastText (0=<unk>, 1=<pad>, 2+=words)
    vocab = {"<unk>": 0, "<pad>": 1}
    for idx, word in enumerate(fasttext_model.wv.index_to_key):
        vocab[word] = idx + 2

    def encode(tokens, vocab):
        return [vocab.get(t, 0) for t in tokens]

    train_texts_tok = [encode(s, vocab) for s in train_tk]
    val_texts_tok   = [encode(s, vocab) for s in val_tk]
    test_texts_tok  = [encode(s, vocab) for s in test_tk]

    # Save cache
    with open(tok_cache, "wb") as f:
        pickle.dump({"train": train_texts_tok, "val": val_texts_tok, "test": test_texts_tok}, f)
    with open(vocab_cache, "w") as f:
        json.dump(vocab, f)

    print(f"Vocab size: {len(vocab)} (saved to cache)")

In [ ]:
#Create Datasets and DataLoader
MAX_LEN    = 2048
BATCH_SIZE = 64
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train_dataset = JobPostingDataset(train_texts_tok, X_train_numeric, y_train, max_len=MAX_LEN)
val_dataset   = JobPostingDataset(val_texts_tok,   X_val_numeric,   y_val,   max_len=MAX_LEN)
test_dataset  = JobPostingDataset(test_texts_tok,  X_test_numeric,  y_test,  max_len=MAX_LEN)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_dataloader)}, Val: {len(val_dataloader)}, Test: {len(test_dataloader)}")

In [ ]:
#Build embedding matrix from FastText
vocab_size = len(vocab)
embed_dim  = fasttext_model.wv.vector_size

embedding_matrix = np.zeros((vocab_size, embed_dim), dtype=np.float32)
for word, idx in vocab.items():
    if word in fasttext_model.wv:
        embedding_matrix[idx] = fasttext_model.wv[word]

pretrained_embeddings = torch.tensor(embedding_matrix, dtype=torch.float)
print(f"Embedding matrix: {pretrained_embeddings.shape}")

In [ ]:
#Instantiate model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FakeJobDetector(
    vocab_size             = vocab_size,
    embed_dim              = embed_dim,
    gru_hidden_dim         = 64,
    num_numerical_features = X_train_numeric.shape[1],
    num_hidden_dim         = 32,
    pretrained_embeddings  = pretrained_embeddings,
    device                 = device,
)
print(f"Device: {device}")
print(model)
print("cuda available:", torch.cuda.is_available())

num_real = sum(1 for y in y_train if y == 0)
num_fake = sum(1 for y in y_train if y == 1)
print(f"Num real: {num_real}, Num fake: {num_fake}")

In [ ]:
#Train
num_real = sum(1 for y in y_train if y == 0)
num_fake = sum(1 for y in y_train if y == 1)

train_losses, val_losses = model.fit(
    dataloader     = train_dataloader,
    val_dataloader = val_dataloader,
    num_epochs     = 20,
    learning_rate  = 1e-3,
    save_path      = "best_model_v5.pt",
    #pos_weight     = 3.0, #math.sqrt(num_real / num_fake), #rea/fake is 18.6, sqrt is 4.3
)


In [ ]:
#Evaluate on test set
for threshold in [0.2, 0.3, 0.4, 0.5, 0.55, 0.6, 0.7, 0.8]:
    print(f"\n--- Threshold: {threshold} ---")
    model.evaluate(test_dataloader, threshold=threshold)